In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Load the dataset
df = pd.read_csv("../data/raw/youtoxic_english_1000.csv")

In [ ]:
df.head()


In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

## Distribución de Clases

In [ ]:
# Columnas de etiquetas
label_columns = ['IsToxic', 'IsAbusive', 'IsThreat', 'IsProvocative', 'IsObscene', 
                'IsHatespeech', 'IsRacist', 'IsNationalist', 'IsSexist', 
                'IsHomophobic', 'IsReligiousHate', 'IsRadicalism']

# Contar valores para cada etiqueta
label_counts = df[label_columns].sum().sort_values(ascending=False)
print("Distribución de etiquetas:")
print(label_counts)
print(f"\nTotal de comentarios: {len(df)}")
print(f"Comentarios tóxicos: {df['IsToxic'].sum()} ({df['IsToxic'].mean()*100:.1f}%)")
print(f"Comentarios no tóxicos: {(1-df['IsToxic']).sum()} ({(1-df['IsToxic'].mean())*100:.1f}%)")

In [ ]:


# Visualización de distribución
plt.figure(figsize=(12, 6))
label_counts.plot(kind='bar', color='steelblue')
plt.title('Distribución de Categorías de Toxicidad', fontsize=14, fontweight='bold')
plt.xlabel('Categoría', fontsize=12)
plt.ylabel('Número de Comentarios', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

 # Análisis de Texto

 util para optimizacion de modelos de NLP(los niveles superiores de nuestro projecto)

In [ ]:
# Longitud de los textos
df['text_length'] = df['Text'].str.len()
df['word_count'] = df['Text'].str.split().str.len()

print("Estadísticas de longitud de texto:")
print(df[['text_length', 'word_count']].describe())

# Comparar longitud entre tóxicos y no tóxicos
print("\nLongitud promedio por categoría:")
print(f"Tóxicos: {df[df['IsToxic']==1]['text_length'].mean():.1f} caracteres")
print(f"No tóxicos: {df[df['IsToxic']==0]['text_length'].mean():.1f} caracteres")

In [ ]:
# Visualización de distribución de longitud
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de longitud de caracteres
axes[0].hist(df[df['IsToxic']==1]['text_length'], bins=30, alpha=0.6, label='Tóxico', color='red')
axes[0].hist(df[df['IsToxic']==0]['text_length'], bins=30, alpha=0.6, label='No tóxico', color='green')
axes[0].set_xlabel('Longitud (caracteres)')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de Longitud de Texto')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Boxplot de número de palabras
df.boxplot(column='word_count', by='IsToxic', ax=axes[1])
axes[1].set_xlabel('IsToxic')
axes[1].set_ylabel('Número de Palabras')
axes[1].set_title('Número de Palabras por Categoría')
plt.suptitle('')

plt.tight_layout()
plt.show()

## Relación entre IsToxic y otras categorías

In [ ]:
# Verificar si IsToxic es la suma lógica de las demás categorías
print("¿IsToxic incluye todas las demás categorías?\n")

# Verificar si un comentario tiene alguna subcategoría pero NO es tóxico
other_toxic_cols = ['IsAbusive', 'IsThreat', 'IsProvocative', 'IsObscene', 
                    'IsHatespeech', 'IsRacist', 'IsNationalist', 'IsSexist', 
                    'IsHomophobic', 'IsReligiousHate', 'IsRadicalism']

# Comentarios que tienen alguna subcategoría = 1
df['has_any_subcategory'] = df[other_toxic_cols].sum(axis=1) > 0

# Comparar con IsToxic
print(f"Comentarios marcados en alguna subcategoría: {df['has_any_subcategory'].sum()}")
print(f"Comentarios marcados como IsToxic: {df['IsToxic'].sum()}")

# ¿Hay comentarios con subcategoría pero sin IsToxic?
subcategory_without_toxic = df[(df['has_any_subcategory'] == True) & (df['IsToxic'] == 0)]
print(f"\n❌ Comentarios con subcategoría pero IsToxic=0: {len(subcategory_without_toxic)}")

# ¿Hay comentarios con IsToxic pero sin subcategoría?
toxic_without_subcategory = df[(df['IsToxic'] == 1) & (df['has_any_subcategory'] == False)]
print(f"❌ Comentarios con IsToxic=1 pero sin subcategoría: {len(toxic_without_subcategory)}")

print("\n" + "="*60)
if len(subcategory_without_toxic) == 0 and len(toxic_without_subcategory) == 0:
    print("✅ IsToxic = OR lógico de todas las subcategorías")
else:
    print("⚠️ IsToxic NO es exactamente la suma de subcategorías")

In [ ]:
# Buscar comentarios con IsToxic=1 pero TODAS las subcategorías en 0
toxic_no_subcategories = df[(df['IsToxic'] == 1) & (df[other_toxic_cols].sum(axis=1) == 0)]

print(f"Comentarios con IsToxic=1 pero todas las subcategorías=0: {len(toxic_no_subcategories)}")

if len(toxic_no_subcategories) > 0:
    print("\n⚠️ SÍ existen comentarios tóxicos sin subcategoría específica")
    print("\nEjemplos:")
    for idx, row in toxic_no_subcategories.head(5).iterrows():
        print(f"\nTexto: {row['Text']}")
        print(f"IsToxic: {row['IsToxic']}")
        print("-" * 80)
else:
    print("\n✅ NO hay comentarios tóxicos sin al menos una subcategoría")

Conclusions:

    We have detected:
    1. A very small dataset - only 1000 comments (462 toxic, 538 non-toxic)
    2. Very imbalanced labels:
        IsToxic            462
        IsAbusive          353
        IsProvocative      161
        IsHatespeech       138
        IsRacist           125
        IsObscene          100
        IsThreat            21 - too few for a robust model
        IsReligiousHate     12 - too few
        IsNationalist        8 - very, very few
        IsSexist             1 - unusable
        IsHomophobic         0 - impossible to train, no class present
        IsRadicalism         0 - impossible to train, no class present
    3. Semantically similar classes: IsToxic, IsAbusive, IsHatespeech, IsProvocative, etc.
    4. "IsToxic" is the logical OR of all the subcategories

    We have a multi-label dataset, and to solve our problem we would need to train a multi-label classification model.
    However, given the findings, I believe it is not feasible to train this type of model with the available data (which is so limited) for several reasons:
    - The model cannot generalize
    - It tends to memorize the few existing cases (overfitting)
    - Or it ignores the small classes and never predicts them (biased predictions)
    - Most labels represent variants of the same phenomenon (toxicity)
Practically, this would produce an unstable and useless model in production.

Solution:
With 1000 comments and 462 toxic examples, the dataset is suitable for training a binary "toxic vs non-toxic" model because:
    - The dataset is less imbalanced: 46.20% toxic / 53.80% non-toxic
    - There are enough examples for both classes
    - It aligns with the client's objective: "a practical solution over a precise tool".

Next steps:
    1. Text preprocessing (cleaning, normalization)
    2. Feature engineering (TF-IDF, embeddings)
    3. Baseline model (Logistic Regression / SVM)
    4. Evaluation and optimization
    5. Deployment (API + interface)